# History feature modls comparison

Core logic same as 3 fatigue_modeling.ipynb.

In [ ]:
%pip install -q -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [15]:
import sys
from pathlib import Path

_src = Path('../../src').resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# Ensure local src edits are picked up when re-running this cell.
for _mod in [k for k in list(sys.modules) if k == 'modeling' or k.startswith('modeling.')]:
    del sys.modules[_mod]

import pandas as pd
from modeling.baselines import (
    run_all_baseline_benchmarks,
    summarize_baseline_metrics,
)
from modeling.config import (
    DATA_PATH,
    HISTORY_CANDIDATE_FEATURES,
    HISTORY_FEATURES,
    N_CV_FOLDS,
    OPTUNA_TRIALS,
    TIME_COL,
    TIME_SERIES_GROUP_COLS,
)
from modeling.data import (
    build_split_bundle,
    load_fatigue_data,
    participant_strata,
    preprocess_after_split,
    split_participant_ids,
    split_summary_table,
)
from modeling.registry import ORDINAL_MODELS
from modeling.runner import tune_and_benchmark_model
from modeling.summaries import (
    CATEGORY_ORDER,
    build_history_ablation_summary,
    build_history_feature_count_comparison,
    collect_categorized_summaries,
    collect_summaries,
)

HISTORY_7_COLS = list(HISTORY_CANDIDATE_FEATURES)
HISTORY_3_COLS = list(HISTORY_FEATURES)

In [ ]:
# Load and preprocess data, then split into train/val/test sets.
df = load_fatigue_data('../../' + DATA_PATH)
df = df.sort_values(TIME_SERIES_GROUP_COLS + [TIME_COL]).reset_index(drop=True)

strata = participant_strata(df)
train_val_ids, test_ids = split_participant_ids(df['id'].unique(), strata=strata)
train_val_mask = df['id'].isin(train_val_ids)
test_mask = df['id'].isin(test_ids)

literacy_col = 'menstrual_health_literacy_num'
print(f'Literacy NaNs before preprocess: {df[literacy_col].isna().sum()}')
df = preprocess_after_split(df, train_val_mask)
print(f'Literacy NaNs after preprocess: {df[literacy_col].isna().sum()}')

bundle = build_split_bundle(df, train_val_ids, test_ids, train_val_mask, test_mask)

print(f"Rows: {len(df):,}  Participants: {df['id'].nunique()}")
display(split_summary_table(bundle))
print('Test participant ids:', sorted(bundle.test_ids))


Literacy NaNs before preprocess: 80
Literacy NaNs after preprocess: 0
Rows: 3,331  Participants: 42


,split,participants,rows,mean_fatigue
0,train_val,34,2659,2.462204
1,test,8,672,2.653274


Test participant ids: [np.int64(7), np.int64(14), np.int64(24), np.int64(38), np.int64(40), np.int64(41), np.int64(46), np.int64(50)]


Re-run the **init accumulators** cell below before a fresh partial run to clear prior tuned-model results.


In [17]:
# Re-run this cell to clear accumulated model results before a fresh partial run.
ordinal_results = []
history7_ordinal_results = []
history3_ordinal_results = []

ordinal_best_params = {}
history7_best_params = {}
history3_best_params = {}

In [ ]:
# Baseline benchmarks (no history features) are run first, so that tuned models can be compared to them.
ordinal_baseline_results = run_all_baseline_benchmarks(bundle, n_splits=N_CV_FOLDS)

ordinal_baseline_summary = summarize_baseline_metrics(ordinal_baseline_results)

print('Ordinal baselines (test metrics)')
display(ordinal_baseline_summary[[c for c in ordinal_baseline_summary.columns if c.startswith('test_')]])


Ordinal baselines (test metrics)


,test_mae,test_rmse,test_r2,test_qwk
model,,,,
global_mean,1.406250,1.640721,-0.188402,0.000000
global_mode,1.156250,1.544479,-0.053072,0.000000
lag1_fatigue,0.950893,1.424175,0.104593,0.549449
expanding_mean,1.025298,1.336863,0.211017,0.422289


MAE: Mean Absolute Error;

RMSE: Root Mean Squared Error, measures the variation in residual/error

R2: how much variability is explained by the model

QWK: Quadratic Weighted Kappa. QWK measures the agreement between two raters—such as an AI and a human—on an ordered scale. It is designed to adjust for chance agreements and heavily penalize larger scoring discrepancies over minor ones.

### History (7 features)

Same seven ordinal models as above, with **all seven history candidate columns** (`HISTORY_CANDIDATE_FEATURES`) appended to the daily feature matrix. History construction uses `EWMA_ALPHA` and `ROLLING_WINDOWS` from `config.py` via `build_split_bundle`; first-day NaNs in history columns are imputed with the train/val median.
> The `EWMA_ALPHA` and `ROLLING_WINDOWS` are optimized from the [`history feature engineering.ipynb`](history%20feature%20engineering.ipynb) notebook.

**History features** (7 cols):
- fatigue lag1: Yesterday's fatigue score
- fatigue EWMA: Exponentially weighted average of past fatigue; recent days count more
- fatigue expanding mean: Average fatigue on all earlier days for this person
- fatigue delta lag1: Change in fatigue, the worsening/improving trend
- activity_logsum_roll_mean: Rolling mean of prior days' sum of log1p(lightly) + log1p(moderately) + log1p(very) (window from ROLLING_WINDOWS)
- calories_sum_roll_mean: Rolling mean of prior daily calories burned (window from ROLLING_WINDOWS)
- very_roll_mean: Rolling mean of prior "very active" minutes (window from ROLLING_WINDOWS)


#### Ordinal Regression (history7)

Continuous loss on `fatigue_num`, then round and clip to [0, 5].


##### `linear_regression` (history7)


In [26]:
_name = 'linear_regression'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_7_COLS,
    display_name=f'{_name}_history7',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history7_ordinal_results.append(_result)
history7_best_params[f'{_name}_history7'] = _params
print(f'[ok] {_name}_history7  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] linear_regression_history7  test_mae=0.8839


##### `ordinal_rf` (history7)


In [27]:
_name = 'ordinal_rf'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_7_COLS,
    display_name=f'{_name}_history7',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history7_ordinal_results.append(_result)
history7_best_params[f'{_name}_history7'] = _params
print(f'[ok] {_name}_history7  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_rf_history7  test_mae=0.8958


##### `catboost_regressor` (history7)


In [28]:
_name = 'catboost_regressor'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_7_COLS,
    display_name=f'{_name}_history7',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history7_ordinal_results.append(_result)
history7_best_params[f'{_name}_history7'] = _params
print(f'[ok] {_name}_history7  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_regressor_history7  test_mae=0.9092


#### Ordinal Classification (history7)


##### `ordered_logistic` (history7)


In [29]:
_name = 'ordered_logistic'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_7_COLS,
    display_name=f'{_name}_history7',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history7_ordinal_results.append(_result)
history7_best_params[f'{_name}_history7'] = _params
print(f'[ok] {_name}_history7  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordered_logistic_history7  test_mae=0.8988


##### `ordinal_forest` (history7)


In [30]:
_name = 'ordinal_forest'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_7_COLS,
    display_name=f'{_name}_history7',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history7_ordinal_results.append(_result)
history7_best_params[f'{_name}_history7'] = _params
print(f'[ok] {_name}_history7  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_forest_history7  test_mae=0.8988


##### `population_ordered_logistic` (history7)


In [31]:
_name = 'population_ordered_logistic'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_7_COLS,
    display_name=f'{_name}_history7',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history7_ordinal_results.append(_result)
history7_best_params[f'{_name}_history7'] = _params
print(f'[ok] {_name}_history7  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] population_ordered_logistic_history7  test_mae=0.8929


##### `catboost_ordinal` (history7)


In [32]:
_name = 'catboost_ordinal'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_7_COLS,
    display_name=f'{_name}_history7',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history7_ordinal_results.append(_result)
history7_best_params[f'{_name}_history7'] = _params
print(f'[ok] {_name}_history7  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_ordinal_history7  test_mae=0.8958


### History (3 features)

Same seven ordinal models with the **FE forward-selected 3-column subset** (`HISTORY_FEATURES` from `config.py`: `fatigue_ewma`, `fatigue_expanding_mean`, `fatigue_lag1`). Same construction params as the 7-feature block above.


#### Ordinal Regression (history3)

Continuous loss on `fatigue_num`, then round and clip to [0, 5].


##### `linear_regression` (history3)


In [33]:
_name = 'linear_regression'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_3_COLS,
    display_name=f'{_name}_history3',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history3_ordinal_results.append(_result)
history3_best_params[f'{_name}_history3'] = _params
print(f'[ok] {_name}_history3  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] linear_regression_history3  test_mae=0.8795


##### `ordinal_rf` (history3)


In [34]:
_name = 'ordinal_rf'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_3_COLS,
    display_name=f'{_name}_history3',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history3_ordinal_results.append(_result)
history3_best_params[f'{_name}_history3'] = _params
print(f'[ok] {_name}_history3  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_rf_history3  test_mae=0.8929


##### `catboost_regressor` (history3)


In [35]:
_name = 'catboost_regressor'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_3_COLS,
    display_name=f'{_name}_history3',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history3_ordinal_results.append(_result)
history3_best_params[f'{_name}_history3'] = _params
print(f'[ok] {_name}_history3  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_regressor_history3  test_mae=0.9226


#### Ordinal Classification (history3)


##### `ordered_logistic` (history3)


In [36]:
_name = 'ordered_logistic'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_3_COLS,
    display_name=f'{_name}_history3',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history3_ordinal_results.append(_result)
history3_best_params[f'{_name}_history3'] = _params
print(f'[ok] {_name}_history3  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordered_logistic_history3  test_mae=0.8869


##### `ordinal_forest` (history3)


In [37]:
_name = 'ordinal_forest'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_3_COLS,
    display_name=f'{_name}_history3',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history3_ordinal_results.append(_result)
history3_best_params[f'{_name}_history3'] = _params
print(f'[ok] {_name}_history3  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_forest_history3  test_mae=0.9122


##### `population_ordered_logistic` (history3)


In [38]:
_name = 'population_ordered_logistic'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_3_COLS,
    display_name=f'{_name}_history3',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history3_ordinal_results.append(_result)
history3_best_params[f'{_name}_history3'] = _params
print(f'[ok] {_name}_history3  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] population_ordered_logistic_history3  test_mae=0.8914


##### `catboost_ordinal` (history3)


In [39]:
_name = 'catboost_ordinal'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    history_cols=HISTORY_3_COLS,
    display_name=f'{_name}_history3',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history3_ordinal_results.append(_result)
history3_best_params[f'{_name}_history3'] = _params
print(f'[ok] {_name}_history3  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_ordinal_history3  test_mae=0.9003


### Results summary

Aggregates baselines plus any models ran (`_history7`, and/or `_history3`). Results are grouped into **baseline** and **history** categories.

The next cell prints **CV** tables in order baseline → history, then **test** tables in the same order. Within each table, rows are sorted by `cv_mae` or `test_mae` respectively. The following cells compare 3-col vs 7-col history test MAE.

In [ ]:
# Merge baselines (§2), base tuned models (§3), and history variants (§3 History).
# globals().get(...) allows partial notebook runs without NameError on skipped cells.

history7_ordinal_results = globals().get('history7_ordinal_results', [])
history3_ordinal_results = globals().get('history3_ordinal_results', [])
history7_best_params = globals().get('history7_best_params', {})
history3_best_params = globals().get('history3_best_params', {})

ran_tuned_models = sorted(
    set(history7_best_params)
    | set(history3_best_params)
)
print(f'Ran {len(ran_tuned_models)} tuned ordinal models: {ran_tuned_models}')

all_ordinal_results = (
    ordinal_baseline_results
    + history7_ordinal_results
    + history3_ordinal_results
)

ordinal_cv_summary, ordinal_test_summary = collect_summaries(all_ordinal_results)
category_summaries = collect_categorized_summaries(all_ordinal_results)

print('CV (sorted by cv_mae within each category; cv_* = mean over GroupKFold folds on train/val)')
for category in CATEGORY_ORDER:
    cv_cat, _ = category_summaries[category]
    if cv_cat.empty:
        continue
    print(f'  {category}')
    display(cv_cat)

print('Test (sorted by test_mae within each category; refit on full train/val, scored on test participants)')
for category in CATEGORY_ORDER:
    _, test_cat = category_summaries[category]
    if test_cat.empty:
        continue
    print(f'  {category}')
    display(test_cat)

Ran 21 tuned ordinal models: ['catboost_ordinal', 'catboost_ordinal_history3', 'catboost_ordinal_history7', 'catboost_regressor', 'catboost_regressor_history3', 'catboost_regressor_history7', 'linear_regression', 'linear_regression_history3', 'linear_regression_history7', 'ordered_logistic', 'ordered_logistic_history3', 'ordered_logistic_history7', 'ordinal_forest', 'ordinal_forest_history3', 'ordinal_forest_history7', 'ordinal_rf', 'ordinal_rf_history3', 'ordinal_rf_history7', 'population_ordered_logistic', 'population_ordered_logistic_history3', 'population_ordered_logistic_history7']
CV (sorted by cv_mae within each category; cv_* = mean over GroupKFold folds on train/val)
  baseline


,best_params,cv_mae,cv_rmse,cv_r2,cv_qwk,cv_mae_std
model,,,,,,
lag1_fatigue,{},0.824015,1.315205,0.096947,0.546287,0.193110
expanding_mean,{},0.867974,1.198179,0.280799,0.495990,0.127344
global_mode,{},1.216046,1.554387,-0.200378,0.000000,0.243084
global_mean,{},1.348516,1.606173,-0.297646,0.000000,0.153930


  base


,best_params,cv_mae,cv_rmse,cv_r2,cv_qwk,cv_mae_std
model,,,,,,
catboost_ordinal,"{'iterations': 102, 'depth': 8, 'learning_rate...",1.183196,1.449413,-0.066497,0.128025,0.049470
ordinal_forest,"{'n_estimators': 101, 'max_depth': 14, 'min_sa...",1.225908,1.495886,-0.118638,0.121176,0.125449
catboost_regressor,"{'iterations': 113, 'depth': 8, 'learning_rate...",1.241518,1.508876,-0.149687,0.084885,0.156451
ordinal_rf,"{'n_estimators': 497, 'max_depth': 17, 'min_sa...",1.289786,1.602694,-0.272464,0.060265,0.178878
population_ordered_logistic,{'maxiter': 424},1.349075,1.702819,-0.486956,0.020996,0.092835
linear_regression,{'alpha': 9.933986358189298},1.465300,1.731218,-0.519044,-0.003901,0.240018
ordered_logistic,{'alpha': 9.75298397130566},1.546936,1.826368,-0.717851,-0.014253,0.213686


  history


...

,best_params,test_mae,test_rmse,test_r2,test_qwk
model,,,,,
lag1_fatigue,{},0.950893,1.424175,0.104593,0.549449
expanding_mean,{},1.025298,1.336863,0.211017,0.422289
global_mode,{},1.156250,1.544479,-0.053072,0.000000
global_mean,{},1.406250,1.640721,-0.188402,0.000000


  base


,best_params,test_mae,test_rmse,test_r2,test_qwk
model,,,,,
catboost_ordinal,"{'iterations': 102, 'depth': 8, 'learning_rate...",1.156250,1.483541,0.028388,0.111997
catboost_regressor,"{'iterations': 113, 'depth': 8, 'learning_rate...",1.200893,1.512350,-0.009714,0.078531
ordinal_forest,"{'n_estimators': 101, 'max_depth': 14, 'min_sa...",1.214286,1.538204,-0.044532,0.059677
ordered_logistic,{'alpha': 9.75298397130566},1.309524,1.642987,-0.191686,-0.005966
linear_regression,{'alpha': 9.933986358189298},1.345238,1.622021,-0.161467,-0.004347
ordinal_rf,"{'n_estimators': 497, 'max_depth': 17, 'min_sa...",1.409226,1.722142,-0.309278,-0.038298
population_ordered_logistic,{'maxiter': 424},1.455357,1.903162,-0.598988,-0.095675


  history


,best_params,test_mae,test_rmse,test_r2,test_qwk
model,,,,,
linear_regression_history3,{'alpha': 7.7576022640900115},0.879464,1.203294,0.360799,0.547310
linear_regression_history7,{'alpha': 3.361697776657055},0.883929,1.208846,0.354887,0.544330
ordered_logistic_history3,{'alpha': 5.0947963106432},0.886905,1.242837,0.318098,0.550744
population_ordered_logistic_history3,{'maxiter': 795},0.891369,1.307601,0.245178,0.555760
population_ordered_logistic_history7,{'maxiter': 230},0.892857,1.310443,0.241893,0.553429
ordinal_rf_history3,"{'n_estimators': 416, 'max_depth': 3, 'min_sam...",0.892857,1.223529,0.339120,0.531123
catboost_ordinal_history7,"{'iterations': 131, 'depth': 4, 'learning_rate...",0.895833,1.217433,0.345689,0.547213
ordinal_rf_history7,"{'n_estimators': 330, 'max_depth': 3, 'min_sam...",0.895833,1.224745,0.337806,0.528297
ordered_logistic_history7,{'alpha': 4.105644012824196},0.898810,1.252379,0.307587,0.544397


In [ ]:
# --- 3-col vs 7-col history comparison (test MAE only) ---
# delta_mae_3_minus_7 = history3 - history7; negative means 3-col subset wins.

history_count_comparison = build_history_feature_count_comparison(
    ordinal_test_summary, ORDINAL_MODELS
)
if history_count_comparison.empty:
    print('No paired history7/history3 models found — run both history blocks first.')
else:
    print(
        '3-col vs 7-col history comparison '
        '(delta_mae_3_minus_7 = history3 - history7; negative = 3-col better)'
    )
    display(history_count_comparison)

Base vs 7-col history paired comparison (delta_mae = history7 - base; negative = history helps)


,test_mae_base,test_mae_history7,delta_mae
model,,,
population_ordered_logistic,1.455357,0.892857,-0.562500
ordinal_rf,1.409226,0.895833,-0.513393
linear_regression,1.345238,0.883929,-0.461310
ordered_logistic,1.309524,0.898810,-0.410714
ordinal_forest,1.214286,0.898810,-0.315476
catboost_regressor,1.200893,0.909226,-0.291667
catboost_ordinal,1.156250,0.895833,-0.260417


3-col vs 7-col history comparison (delta_mae_3_minus_7 = history3 - history7; negative = 3-col better)


,test_mae_history7,test_mae_history3,delta_mae_3_minus_7
model,,,
ordered_logistic,0.898810,0.886905,-0.011905
linear_regression,0.883929,0.879464,-0.004464
ordinal_rf,0.895833,0.892857,-0.002976
population_ordered_logistic,0.892857,0.891369,-0.001488
catboost_ordinal,0.895833,0.900298,0.004464
ordinal_forest,0.898810,0.912202,0.013393
catboost_regressor,0.909226,0.922619,0.013393
